In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
women_rate = sum(women)/len(women)
print('% of women who survived', women_rate)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
men_rate = sum(men)/len(men)
print('% of men who survived', men_rate)

In [ ]:
from sklearn.model_selection import train_test_split

y = train_data.Survived

features = ["Pclass", "Sex", "SibSp", "Parch"]

X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def get_accuracy(n_estimators, max_depth, train_X, val_X, train_y, val_y):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=1)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    accuracy = accuracy_score(val_y, preds_val)
    return(accuracy)

In [ ]:
candidate_max_depth = [5, 10, 15, 20, 25, 50, 100, 250]
candidate_num_of_estimations = [5, 10, 15, 20, 25, 50, 100, 250]

best_max_depth = 5
best_estimation = 100
best_accuracy = 0

print("Initial Values:", best_estimation, best_max_depth, best_accuracy)

for max_depth in candidate_max_depth:
    for num_of_estimations in candidate_num_of_estimations:

        my_accuracy = get_accuracy(num_of_estimations, max_depth, train_X, val_X, train_y, val_y)
        print("Current Round", num_of_estimations, max_depth, my_accuracy)
    
        if (my_accuracy > best_accuracy):
            best_accuracy = my_accuracy
            best_max_depth = max_depth
            best_estimation = num_of_estimations
            print("Current Values:", best_estimation, best_max_depth, best_accuracy)

print("Best Choice Is:", best_estimation, best_max_depth, best_accuracy)

In [ ]:
# best parameters are 10, 10
model = RandomForestClassifier(n_estimators=10, max_depth=10, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)
print("Prediction Completed Successfully")

In [ ]:
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)

print("Your submission was successfully saved!")